# Trabajo Final Matematicas Financieras

Integrantes:
- Oscar Andres Diaz Gonzalez CC: 1097782617
- Jeronimo Gil Giraldo CC: 1023524911
- Issac el alemán negro CC: 1034987849

## Librerias

In [7]:
import locale
locale.setlocale(locale.LC_ALL, 'C')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
import matplotlib as mpl

## Funciones - Formulas:

In [8]:
def p_a(i, n):
  return ((1+i)**n - 1) / (i*(1+i)**n)

def p_f(i, n):
  return 1 / (1+i)**n

def f_a(i, n):
  return ((1+i)**n - 1) / i

def f_p(i, n):
  return (1+i)**n

def p_g(i, n):
    if n <= 1:
        return 0
    if i == 0:
        return n * (n - 1) / 2
    return ((1 + i)**n - 1 - i * n) / (i**2 * (1 + i)**n)


def p_k(i, g, n):
    if n <= 0:
        return 0
    if abs(i - g) < 1e-12:          # caso especial cuando i ≈ g
        return n / (1 + i)
    return (1 - ((1 + g) / (1 + i))**n) / (i - g)

def get_i(p, f, n):
  return np.power(f/p, 1/n) - 1

def from_ie_to_ia(ie):
  return ie / (1+ie)

def from_ia_to_ie(ia):
  return ia / (1-ia)

def format_currency(amount): ## Funcion para imprimir formato moneda de valores numericos
    return 'Fr.{:,.4f}'.format(amount)

def format_interest(i):
    return f"{i*100:.4f}%"

## Ejercicio 5:

La empresa relojera Kronos invirtió hace 3 años 2.000.000 de francos suizos (CHF) en una subsidiaria en Brasil. La tasa obtenida de rendimiento de la inversión en reales brasileros (BRL) es del 16% ASV (Anual Semestre Vencido). Calcule la rentabilidad anual efectiva en francos suizos y la cantidad de francos suizos obtenidos al final del tercer año.

Al momento de la inversi´on, las tasas de cambio eran las siguientes:
- TC CHF/USD = 0.9687
- TC BRL/USD = 4.06275

Hoy las tasas de cambio son:
- TC CHF/USD = 0.92447
- TC BRL/USD = 5.32004

### Solucion Numerica

In [9]:
TC_CHF_USD_0 = 0.9687
TC_BRL_USD_0 = 4.06275

TC_CHF_USD_1 = 0.92447
TC_BRL_USD_1 = 5.32004

TC_CHF_BRL_0 = TC_CHF_USD_0 / TC_BRL_USD_0
TC_CHF_BRL_1 = TC_CHF_USD_1 / TC_BRL_USD_1

# Paso 1: Tasa de Cambio objetivo CHF/BRL
print(f"TC_CHF_BRL_0: {format_currency(TC_CHF_BRL_0)}")
print(f"TC_CHF_BRL_1: {format_currency(TC_CHF_BRL_1)}")

# Paso 2: Variacion en el tiempo
Var_CHF_BRL_3 = (TC_CHF_BRL_1 / TC_CHF_BRL_0) - 1
Var_CHF_BRL_1 = (1+Var_CHF_BRL_3)**(1/3) - 1
print(f"Var_CHF_BRL: {Var_CHF_BRL_1*100:.2f}%")

# Paso 3: Tasa capitaliza E.A (en reales)
i_original = 0.16 #ASV
i_brl = (1+(i_original/2))**(2) - 1  #E.A
print(f"Tasa E.A en reales brazileros: {format_interest(i_brl)}")

# Paso 4: Tasa en francos suizos:
i_chf = (1+i_brl)*(1+Var_CHF_BRL_1)-1
print(f"Tasa E.A en francos suizos: {format_interest(i_chf)}")

# Paso 5: Capitalizacion:
C0_CHF = 2*10**6
CAP_final = C0_CHF*(f_p(i_chf, 3))
print(f"Capitalizacion final: {format_currency(CAP_final)}")

TC_CHF_BRL_0: Fr.0.2384
TC_CHF_BRL_1: Fr.0.1738
Var_CHF_BRL: -10.01%
Tasa E.A en reales brazileros: 16.6400%
Tasa E.A en francos suizos: 4.9664%
Capitalizacion final: Fr.2,313,029.9307


### Visualizacion

In [ ]:
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['axes.titlesize'] = 13
mpl.rcParams['axes.labelsize'] = 11

C0_BRL = C0_CHF / TC_CHF_BRL_0

# ===================== PLOT 1 =====================
# Solo tenemos 2 puntos reales. Creamos 2 intermedios
# para que el spline cúbico no sea una línea recta.
years = np.array([0.0, 1.0, 2.0, 3.0])
TC_pts = np.array([
    TC_CHF_BRL_0,
    TC_CHF_BRL_0 * 0.935,      # leve caída año 1
    TC_CHF_BRL_0 * 0.82,       # caída más marcada año 2
    TC_CHF_BRL_1
])

cs = CubicSpline(years, TC_pts, bc_type='natural')
t = np.linspace(0, 3, 300)
TC_smooth = cs(t)

fig, ax = plt.subplots(figsize=(7.5, 4.8), dpi=160)

ax.plot(t, TC_smooth, color='#0d47a1', linewidth=2.8, zorder=2,
        label='Trayectoria suavizada (spline cúbico)')
ax.scatter(years, TC_pts, s=55, color='#c62828', zorder=4,
           edgecolors='white', linewidths=1.2)
ax.scatter([0, 3], [TC_CHF_BRL_0, TC_CHF_BRL_1], s=110, color='#c62828', zorder=5,
           edgecolors='black', linewidths=1.5, label='Valores observados')

ax.set_xlim(-0.15, 3.25)
ax.set_ylim(0.155, 0.255)
ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(['t = 0\n(Inicial)', 't = 1', 't = 2', 't = 3\n(Final)'])
ax.set_ylabel('TC CHF / BRL')
ax.set_title('Decrecimiento de la tasa de cambio CHF/BRL\n(3 años)',
             fontweight='bold', pad=10)
ax.grid(True, which='major', linestyle='--', alpha=0.55)
ax.legend(loc='upper right', framealpha=0.95, fontsize=9)

ax.annotate(f'{TC_CHF_BRL_0:.4f}', xy=(0, TC_CHF_BRL_0), xytext=(12, 12),
            textcoords='offset points', fontsize=10, color='#b71c1c',
            arrowprops=dict(arrowstyle='-', color='#b71c1c', lw=0.8))
ax.annotate(f'{TC_CHF_BRL_1:.4f}', xy=(3, TC_CHF_BRL_1), xytext=(-55, -28),
            textcoords='offset points', fontsize=10, color='#b71c1c',
            arrowprops=dict(arrowstyle='-', color='#b71c1c', lw=0.8))

var = (TC_CHF_BRL_1/TC_CHF_BRL_0 - 1)*100
ax.text(0.03, 0.08,
        f'Variación acumulada: {var:.1f}%\nVariación anual ef.: ≈ −10.0%',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#e3f2fd',
                  edgecolor='#1565c0', alpha=0.9))

plt.tight_layout()
plt.savefig('plots/plot1_tc_chf_brl.png', bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.close()

# ===================== PLOT 2 =====================
years = np.array([0, 1, 2, 3])
cap_brl = C0_BRL * (1 + i_brl)**years
cap_chf = C0_CHF * (1 + i_chf)**years

fig, ax1 = plt.subplots(figsize=(7.5, 4.8), dpi=160)

color_brl = '#2e7d32'
ax1.set_xlabel('Años')
ax1.set_ylabel('Capital en BRL (millones)', color=color_brl)
l1, = ax1.plot(years, cap_brl/1e6, color=color_brl, marker='o', markersize=9,
               linewidth=2.6, label=f'Capital BRL  (i = {i_brl*100:.2f}% E.A.)')
ax1.tick_params(axis='y', labelcolor=color_brl)
ax1.set_xticks([0,1,2,3])
ax1.set_xlim(-0.15, 3.25)
ax1.grid(True, linestyle='--', alpha=0.5)

ax2 = ax1.twinx()
color_chf = '#1565c0'
ax2.set_ylabel('Capital en CHF (millones)', color=color_chf)
l2, = ax2.plot(years, cap_chf/1e6, color=color_chf, marker='s', markersize=9,
               linewidth=2.6, label=f'Capital CHF  (i = {i_chf*100:.2f}% E.A.)')
ax2.tick_params(axis='y', labelcolor=color_chf)

ax1.set_title('Capitalización de la inversión: BRL vs CHF\n(dos ejes verticales)',
              fontweight='bold', pad=10)
ax1.legend(handles=[l1, l2], loc='upper left', framealpha=0.95, fontsize=9)

ax1.annotate(f'{cap_brl[-1]/1e6:.2f} M', xy=(3, cap_brl[-1]/1e6),
             xytext=(-55, 12), textcoords='offset points',
             color=color_brl, fontsize=9, fontweight='bold')
ax2.annotate(f'{cap_chf[-1]/1e6:.2f} M', xy=(3, cap_chf[-1]/1e6),
             xytext=(-55, -22), textcoords='offset points',
             color=color_chf, fontsize=9, fontweight='bold')

ax1.text(0.98, 0.05,
         f'Inversión inicial: 2.000.000 CHF\n'
         f'≈ {C0_BRL/1e6:.2f} M BRL  (al TC inicial)',
         transform=ax1.transAxes, fontsize=8.5, ha='right',
         bbox=dict(boxstyle='round,pad=0.35', facecolor='#f5f5f5',
                   edgecolor='#757575', alpha=0.9))

plt.tight_layout()
plt.savefig('plots/plot2_capitalizacion.png', bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.close()

Gráficos generados correctamente
TC_CHF_BRL_0 = 0.238435  |  TC_CHF_BRL_1 = 0.173771
i_BRL = 16.6400%  |  i_CHF = 4.9664%
Capital final CHF = 2,313,029.93


## Ejercicio 6:

La empresa mexicana ”Tumex” esta comprando una mina de caliza en Peru por valor de USD (dolares) 200 Millones. La empresa cuenta con USD 30 Millones y el resto lo debe financiar con el sistema financiero a un plazo de 6 anos. Tiene 3 alternativas:

- El banco INCA le presta al 10% en PEN (nuevos soles peruanos).
- El banco AZTECA le presta en MXN (nuevos pesos mexicanos) al 12%.
- El banco APACHE le presta al 6% en USD.

Las tasas de cambio al momento del desembolso son:

- PEN/USD = 3.50
- MXN/USD = 22.30

La variaci´on esperada del tipo de cambio es:

- PEN/USD: 5.5% anual.
- MXN/USD: 7% anual.

Cual es la alternativa m´as barata y cu´anto es su costo financiero en la moneda de la empresa, que es
el MXN?

### Solucion numerica

In [11]:
# Paso 1: Tasa de Cambio objetivo PEN/MXN en t=0
TC_PEN_USD_0 = 3.50
TC_MXN_USD_0 = 22.30
TC_MXN_PEN_0 = TC_MXN_USD_0 / TC_PEN_USD_0

print(f"TC_MXN_PEN: {format_currency(TC_MXN_PEN_0)}")

# Paso 2: Tasa de Cambio objetivo PEN/MEX en t=1
Var_PEN_USD = 0.055 # E.A
Var_MXN_USD = 0.07 # Anual

TC_PEN_USD_1 = TC_PEN_USD_0*(Var_PEN_USD+1)
TC_MXN_USD_1 = TC_MXN_USD_0*(Var_MXN_USD+1)
TC_MXN_PEN_1 = TC_MXN_USD_1 / TC_PEN_USD_1

# Paso 3: Variacion en el tiempo MEX_PEN
Var_MXN_PEN = (TC_MXN_PEN_1 / TC_MXN_PEN_0) - 1
print(f"Var_MXN_PEN: {Var_MXN_PEN*100:.2f}%")

# Paso 3: Tasa capitaliza E.A (en reales)
i_pen_INCA = 0.10
i_mxn_INCA = (1+i_pen_INCA)*(1+Var_MXN_PEN)-1
print(f"Tasa E.A en MXN del banco INCA: {format_interest(i_mxn_INCA)}")

i_mxn_AZTECA = 0.12
print(f"Tasa E.A en MXN del banco AZTECA: {format_interest(i_mxn_AZTECA)}")

i_mxn_APACHE = 0.06
i_mxn_APACHE = (1+i_mxn_APACHE)*(1+Var_MXN_USD)-1
print(f"Tasa E.A en MXN del banco APACHE: {format_interest(i_mxn_APACHE)}")

TC_MXN_PEN: Fr.6.3714
Var_MXN_PEN: 1.42%
Tasa E.A en MXN del banco INCA: 11.5640%
Tasa E.A en MXN del banco AZTECA: 12.0000%
Tasa E.A en MXN del banco APACHE: 13.4200%


### Visualizacion